# Buổi 22 — Lab

Chạy từng ô từ trên xuống. Mỗi bước ứng với một mục trong tài liệu (mục 5). Bạn sửa `hoi_quy.py`; các ô tự dùng bản mới.

In [ ]:
# sửa tệp .py trong code/ thì các ô sau tự dùng bản mới, không cần khởi động lại
%load_ext autoreload
%autoreload 2

## Bước 1 — Dữ liệu và một chuỗi (mục 4.1)

In [ ]:
%matplotlib inline
import warnings

import hoi_quy as hq
import numpy as np

warnings.simplefilter("ignore")
df = hq.doc_web_traffic()                     # 10.000 trang, cột z = log(1 + lượt xem)
print("số trang:", df["unique_id"].nunique(), "| số ngày:", df["ds"].nunique(), "| cutoff:", [str(c)[:10] for c in hq.cac_cutoff(df)])

X, y = hq.bang_lag(np.array([10.0, 12, 11, 13, 15]), 2)   # ví dụ tay mục 4.1
print(X, y)

# một trang, rừng ngẫu nhiên LOCAL trên 28 lag, recursive 14 ngày ở cutoff đầu
t1 = df[df["unique_id"] == "T1"].reset_index(drop=True)
c = int(np.flatnonzero(t1["ds"] == hq.cac_cutoff(df)[0])[0])
z = t1["z"].to_numpy()
X, y = hq.bang_lag(z[:c + 1], hq.SO_LAG)
du_bao = hq.du_bao_de_quy(hq.rung().fit(X, y), z[:c + 1], hq.H, hq.SO_LAG)
that = z[c + 1:c + 1 + hq.H]
snaive = np.tile(z[c - 6:c + 1], 2)[:hq.H]
print("T1: số dòng bảng", len(y), "| RMSE rừng", round(float(np.sqrt(np.mean((that - du_bao) ** 2))), 3),
      "| RMSE seasonal naive", round(float(np.sqrt(np.mean((that - snaive) ** 2))), 3))

## Bước 2 — Ba chiến lược (mục 4.3–4.4)

300 trang, khoảng 1–3 phút. Sửa `dac_trung` rồi chạy lại ô này.

In [ ]:
df300 = df[df["unique_id"].isin(df["unique_id"].unique()[:300])]
kq = hq.chien_luoc(df300)
kq["tuan"] = np.where(kq["buoc_h"] <= 7, 1, 2)
print(kq.groupby("tuan").apply(lambda d: np.sqrt(((d[["recursive", "direct", "mimo"]].sub(d["y"], axis=0)) ** 2).mean())).round(3))
hq.sai_so_theo_h(kq).plot(marker="o", ylabel="RMSE trên thang z", xlabel="bước h");

## Bước 3 — Cây và xu hướng (mục 4.5)

Sửa `du_bao_cay` rồi chạy lại ô này.

In [ ]:
rng = np.random.default_rng(0)
y = 50 + 2 * np.arange(120) + rng.normal(0, 3, 120)       # chuỗi tăng 2 mỗi ngày
p = hq.du_bao_cay(y, 14)
print("số lớn nhất đã thấy:", round(y.max(), 1), "| dự báo ngày cuối:", round(p[-1], 1), "| xu hướng thật:", 50 + 2 * 133)

## Bước 4 — Global và local trên 10.000 trang (mục 4.6)

Lần đầu 5–15 phút tuỳ máy; kết quả lưu vào `lab/du-lieu/cache/`, lần sau đọc lại ngay.

In [ ]:
kq = hq.backtest_toan_cuc(df, luu=True).merge(hq.backtest_cuc_bo(df, luu=True).drop(columns="z"),
                                             on=["unique_id", "ds", "cutoff"])
r = hq.rmsse_tung_chuoi(kq, df, ["LightGBM", "AutoETS", "SeasonalNaive", "Naive"])
print(hq.bang_so_sanh(r).round(3).to_string())
print("% trang LightGBM thắng AutoETS:", round(float((r["LightGBM"] < r["AutoETS"]).mean() * 100), 1))

## Bước 5 — Kiểm tra

Trong terminal ở thư mục `lab/`: `python lab.py check` — phải xanh 10/10.